# keyboard_config

> Shared keyboard navigation configuration for the combined Phase 2 step

In [ ]:
#| default_exp components.keyboard_config

In [ ]:
#| export
from typing import Any, Tuple, Optional

from fasthtml.common import Div, Script

# Tailwind utilities
from cjm_fasthtml_tailwind.core.base import combine_classes

# Keyboard navigation library
from cjm_fasthtml_keyboard_navigation.core.manager import ZoneManager
from cjm_fasthtml_keyboard_navigation.core.actions import KeyAction
from cjm_fasthtml_keyboard_navigation.components.system import render_keyboard_system

# Card stack library
from cjm_fasthtml_card_stack.keyboard.actions import build_card_stack_url_map

# Local HTML IDs
from cjm_transcript_segment_align.html_ids import CombinedHtmlIds

# Segmentation-specific keyboard config (building blocks)
from cjm_transcript_segmentation.components.keyboard_config import (
    SD_SEG_ENTER_SPLIT_BTN, SD_SEG_EXIT_SPLIT_BTN,
    SD_SEG_SPLIT_BTN, SD_SEG_MERGE_BTN, SD_SEG_UNDO_BTN,
    create_seg_kb_parts,
)
from cjm_transcript_segmentation.components.card_stack_config import (
    SEG_CS_CONFIG, SEG_CS_IDS, SEG_CS_BTN_IDS,
    SEG_TS_IDS,
)

# Alignment-specific keyboard config (building blocks)
from cjm_transcript_vad_align.components.keyboard_config import (
    create_align_kb_parts,
)
from cjm_transcript_vad_align.components.card_stack_config import (
    ALIGN_CS_CONFIG, ALIGN_CS_IDS, ALIGN_CS_BTN_IDS,
)

# URL bundles
from cjm_transcript_segmentation.models import SegmentationUrls
from cjm_transcript_vad_align.models import AlignmentUrls

# Sync controls
from cjm_transcript_segment_align.components.sync_controls import (
    generate_sync_key_toggle_js,
)

# Debug flag for keyboard system tracing (set False in production)
DEBUG_KB_SYSTEM = True

## Keyboard Hints (removed)

Keyboard hints now use `render_keyboard_hints_modal()` from `cjm-fasthtml-keyboard-navigation`.


In [ ]:
# render_keyboard_hints_collapsible removed — replaced by render_keyboard_hints_modal


## Keyboard System Builder

Builds the combined keyboard system with both decomposition and alignment zones.
Returns `(manager, system)` tuple where manager is needed for hints rendering
and system contains the script/inputs/buttons for the stable KB container.

The combined KB system is always built with both zones, regardless of which
column has initialized first. Zone switching works immediately; zones with
no DOM content yet will simply have no focusable items until content loads.

In [ ]:
#| export
# Zone change callback name — global JS function called when zone switches
ZONE_CHANGE_CALLBACK = "onCombinedZoneChange"

# Stable keyboard system ID for coordinator pause/resume
KB_SYSTEM_ID = "sd-seg-align-kb"


def build_combined_kb_system(
    seg_urls:SegmentationUrls,  # URL bundle for segmentation routes
    align_urls:AlignmentUrls,  # URL bundle for alignment routes
) -> Tuple[ZoneManager, Any]:  # (keyboard manager, rendered keyboard system)
    """Build combined keyboard system with segmentation and alignment zones."""
    if DEBUG_KB_SYSTEM:
        print("[KB_SYSTEM] build_combined_kb_system called")

    # Get segmentation-specific building blocks
    seg_zone, seg_actions, seg_modes = create_seg_kb_parts(
        ids=SEG_CS_IDS,
        button_ids=SEG_CS_BTN_IDS,
        config=SEG_CS_CONFIG,
    )

    # Get alignment-specific building blocks
    align_zone, align_actions, align_modes = create_align_kb_parts(
        ids=ALIGN_CS_IDS,
        button_ids=ALIGN_CS_BTN_IDS,
        config=ALIGN_CS_CONFIG,
    )

    if DEBUG_KB_SYSTEM:
        print(f"[KB_SYSTEM] seg_zone.id: {seg_zone.id}")
        print(f"[KB_SYSTEM] align_zone.id: {align_zone.id}")
        print(f"[KB_SYSTEM] seg_actions count: {len(seg_actions)}")
        print(f"[KB_SYSTEM] align_actions count: {len(align_actions)}")

    # Synced navigation toggle — S key (active in seg zone only, not in split mode)
    sync_action = KeyAction(
        key="s",
        js_callback="_segAlignSyncKeyToggle",
        zone_ids=(seg_zone.id,),
        not_modes=("split",),
        description="Toggle synced navigation",
        hint_group="Sync",
    )

    # Combine modes (only segmentation has split mode)
    all_modes = (*seg_modes, *align_modes)

    # Combine actions from both zones + sync
    all_actions = (*seg_actions, *align_actions, sync_action)

    # Assemble into ZoneManager with zone switching enabled
    kb_manager = ZoneManager(
        zones=(seg_zone, align_zone),
        actions=all_actions,
        modes=all_modes,
        initial_zone_id=seg_zone.id,
        prev_zone_key="ArrowLeft",
        next_zone_key="ArrowRight",
        on_zone_change=ZONE_CHANGE_CALLBACK,
        state_hidden_inputs=True,
        system_id=KB_SYSTEM_ID,
    )

    # Build URL maps for both stacks
    seg_include_selector = (
        f"#{SEG_CS_IDS.focused_index_input}, "
        f"#{SEG_TS_IDS.anchor_input}"
    )
    align_include_selector = f"#{ALIGN_CS_IDS.focused_index_input}"

    # Segmentation URL mappings
    seg_url_map = {
        **build_card_stack_url_map(SEG_CS_BTN_IDS, seg_urls.card_stack),
        SD_SEG_ENTER_SPLIT_BTN: seg_urls.enter_split,
        SD_SEG_EXIT_SPLIT_BTN: seg_urls.exit_split,
        SD_SEG_SPLIT_BTN: seg_urls.split,
        SD_SEG_MERGE_BTN: seg_urls.merge,
        SD_SEG_UNDO_BTN: seg_urls.undo,
    }

    # Alignment URL mappings (navigation only — no undo for alignment)
    align_url_map = {
        **build_card_stack_url_map(ALIGN_CS_BTN_IDS, align_urls.card_stack),
    }

    # Combined URL map
    url_map = {**seg_url_map, **align_url_map}

    # Target maps (each stack targets its own card_stack element)
    seg_target = f"#{SEG_CS_IDS.card_stack}"
    align_target = f"#{ALIGN_CS_IDS.card_stack}"
    target_map = {
        **{btn_id: seg_target for btn_id in seg_url_map},
        **{btn_id: align_target for btn_id in align_url_map},
    }

    # Include maps (each stack includes its own focused_index input)
    include_map = {
        **{btn_id: seg_include_selector for btn_id in seg_url_map},
        **{btn_id: align_include_selector for btn_id in align_url_map},
    }

    # Swap map (none for all — OOB swaps handle updates)
    swap_map = {btn_id: "none" for btn_id in url_map}

    kb_system = render_keyboard_system(
        kb_manager,
        url_map=url_map,
        target_map=target_map,
        include_map=include_map,
        swap_map=swap_map,
        show_hints=False,
        include_state_inputs=True,
    )

    if DEBUG_KB_SYSTEM:
        print(f"[KB_SYSTEM] combined kb_system built successfully")

    return kb_manager, kb_system

## Zone Change JavaScript

Generates JavaScript for zone change handling including:
- Global callback function that runs when zones switch via keyboard
- Visual styling updates (ring/opacity on columns)
- Hidden input updates for active column state
- Chrome swap trigger via HTMX button click
- Click handlers on columns for mouse-driven zone switching

In [ ]:
#| export
# Hidden button ID for chrome swap HTMX trigger
SWITCH_CHROME_BTN_ID = "sd-switch-chrome-btn"


def generate_zone_change_js(
    switch_chrome_url:str="",  # URL for chrome swap handler (empty = no swap)
) -> Script:  # Script element with zone change callback and click handlers
    """Generate JavaScript for zone change handling and column click handlers."""
    # Zone IDs from card stack configs
    seg_zone_id = SEG_CS_IDS.card_stack
    align_zone_id = ALIGN_CS_IDS.card_stack

    # Card stack prefixes for syncCountDropdown calls
    seg_prefix = SEG_CS_CONFIG.prefix
    align_prefix = ALIGN_CS_CONFIG.prefix

    # Column container IDs
    seg_col_id = CombinedHtmlIds.SEG_COLUMN
    align_col_id = CombinedHtmlIds.ALIGNMENT_COLUMN
    active_input_id = CombinedHtmlIds.ACTIVE_COLUMN_INPUT

    # Chrome swap trigger with dropdown sync (optional)
    chrome_swap_js = ""
    if switch_chrome_url:
        chrome_swap_js = f"""
            // Trigger chrome swap via hidden HTMX button
            const chromeSwitchBtn = document.getElementById('{SWITCH_CHROME_BTN_ID}');
            if (chromeSwitchBtn) {{
                // Add one-time listener to sync dropdown after chrome swap settles
                function onChromeSettle(evt) {{
                    // Sync the active card stack's dropdown from localStorage
                    const activePrefix = (newZoneId === seg_zone_id) ? '{seg_prefix}' : '{align_prefix}';
                    if (window.cardStacks?.[activePrefix]?.syncCountDropdown) {{
                        window.cardStacks[activePrefix].syncCountDropdown();
                    }}
                    document.body.removeEventListener('htmx:afterSettle', onChromeSettle);
                }}
                document.body.addEventListener('htmx:afterSettle', onChromeSettle);
                chromeSwitchBtn.click();
            }}
        """

    # Sync key toggle wrapper JS (S key → toggle sync + update button)
    sync_key_js = generate_sync_key_toggle_js()

    js_code = f"""
    (function() {{
        const seg_zone_id = '{seg_zone_id}';
        const align_zone_id = '{align_zone_id}';
        const seg_col_id = '{seg_col_id}';
        const align_col_id = '{align_col_id}';
        const active_input_id = '{active_input_id}';

        function updateColumnStyles(activeZoneId) {{
            const segCol = document.getElementById(seg_col_id);
            const alignCol = document.getElementById(align_col_id);
            const activeInput = document.getElementById(active_input_id);

            if (!segCol || !alignCol) return;

            const isSegActive = activeZoneId === seg_zone_id;

            // Ring classes for active indicator
            segCol.classList.toggle('ring-1', isSegActive);
            segCol.classList.toggle('ring-primary', isSegActive);
            alignCol.classList.toggle('ring-1', !isSegActive);
            alignCol.classList.toggle('ring-secondary', !isSegActive);

            // Opacity for inactive column
            segCol.classList.toggle('opacity-70', !isSegActive);
            alignCol.classList.toggle('opacity-70', isSegActive);

            // Update hidden input
            if (activeInput) {{
                activeInput.value = isSegActive ? 'seg' : 'align';
            }}
        }}

        // Global callback for ZoneManager (called on keyboard zone switch)
        window.{ZONE_CHANGE_CALLBACK} = function(newZoneId, prevZoneId) {{
            // Skip redundant chrome swap when zone hasn't actually changed
            // (scrollbar/scrollwheel call setActiveZone on every event as a safety measure)
            if (newZoneId === prevZoneId) return;

            if (window.DEBUG_KB_SYSTEM) {{
                console.log('[KB_SYSTEM] Zone change:', prevZoneId, '->', newZoneId);
            }}
            updateColumnStyles(newZoneId);
            {chrome_swap_js}
        }};

        // Sync key toggle wrapper (S key → toggle + update button styling)
        {sync_key_js}

        // Click handler for column focus switching
        function handleColumnClick(targetZoneId) {{
            // Skip if already active
            const state = window.kbNav?.getState?.();
            if (state && state.activeZoneId === targetZoneId) return;

            // Skip if in split mode (don't allow accidental zone switch)
            if (state && state.currentMode === 'split') return;

            // Switch zone using keyboard navigation library API
            if (window.kbNav?.setActiveZone) {{
                window.kbNav.setActiveZone(targetZoneId);
            }}
        }}

        // Set up click handlers on columns
        function setupClickHandlers() {{
            const segCol = document.getElementById(seg_col_id);
            const alignCol = document.getElementById(align_col_id);

            if (segCol) {{
                segCol.addEventListener('mousedown', function(e) {{
                    handleColumnClick(seg_zone_id);
                }});
            }}

            if (alignCol) {{
                alignCol.addEventListener('mousedown', function(e) {{
                    handleColumnClick(align_zone_id);
                }});
            }}
        }}

        // Initial update (segmentation zone is active by default)
        updateColumnStyles(seg_zone_id);

        // Set up click handlers
        setupClickHandlers();
    }})();
    """

    return Script(js_code)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()